# Ames Housing: ML EDA

Dataset: Ames, Iowa home sales 2006-2010 (OpenML #42165, De Cock 2011). Target: `SalePrice`.


## 1.1 Load and look

In [2]:
import pandas as pd
#show all rows and columns
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [3]:

df = pd.read_csv("data/ames_house_prices.csv", keep_default_na=False, na_values=[""])
print(df.shape)   # expected: (1460, 81)
df.head(3)
df.reset_index(drop=True, inplace=True)

(1460, 81)


/var/folders/9t/d8g879bj5ss5s1jty3frvvww0000gn/T/ipykernel_3999/966974404.py:4: Pandas4Warning: The inplace keyword in DataFrame.reset_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.reset_index(drop=True, inplace=True)


In [4]:
table1 = pd.DataFrame({'columns': [col for col in df.columns], 'data_types': [df[col].dtype for col in df.columns], 'missing_values': [df[col].isnull().sum() for col in df.columns],'missing_value_percentage': [round(df[col].isnull().sum() / len(df) * 100, 2) for col in df.columns], 'num_unique_values': [df[col].nunique() for col in df.columns], 'num_zero_values': [df[col].isin([0]).sum() for col in df.columns], 'num_empty_strings': [df[col].isin(['']).sum() for col in df.columns],'data_type_category': [df[col].dtype.name for col in df.columns]})
display(table1.sort_values(by=['missing_value_percentage'], ascending=False))

,columns,data_types,missing_values,missing_value_percentage,num_unique_values,num_zero_values,num_empty_strings,data_type_category
72,PoolQC,str,1453,99.52,3,0,0,str
74,MiscFeature,str,1406,96.30,4,0,0,str
6,Alley,str,1369,93.77,2,0,0,str
73,Fence,str,1179,80.75,4,0,0,str
57,FireplaceQu,str,690,47.26,5,0,0,str
3,LotFrontage,float64,259,17.74,110,0,0,float64
59,GarageYrBlt,float64,81,5.55,97,0,0,float64
64,GarageCond,str,81,5.55,5,0,0,str
58,GarageType,str,81,5.55,6,0,0,str
60,GarageFinish,str,81,5.55,3,0,0,str


In [5]:
missing_table = pd.DataFrame({'columns': [col for col in df.columns], 'missing_values': [df[col].isnull().sum() for col in df.columns],'missing_value_percentage': [round(df[col].isnull().sum() / len(df) * 100, 2) for col in df.columns]})
display(missing_table[missing_table['missing_value_percentage'] > 0].sort_values(by=['missing_value_percentage'], ascending=False))

,columns,missing_values,missing_value_percentage
72,PoolQC,1453,99.52
74,MiscFeature,1406,96.30
6,Alley,1369,93.77
73,Fence,1179,80.75
57,FireplaceQu,690,47.26
3,LotFrontage,259,17.74
58,GarageType,81,5.55
59,GarageYrBlt,81,5.55
60,GarageFinish,81,5.55
63,GarageQual,81,5.55


In [6]:
table2= pd.crosstab(df['PoolQC'].isnull(), df['PoolArea'])
display(table2)

PoolArea,0,480,512,519,555,576,648,738
PoolQC,,,,,,,,
False,0,1,1,1,1,1,1,1
True,1453,0,0,0,0,0,0,0


In [7]:
df.PoolQC = df.PoolQC.fillna('No Pool')

In [8]:
missing_table = pd.DataFrame({'columns': [col for col in df.columns], 'missing_values': [df[col].isnull().sum() for col in df.columns],'missing_value_percentage': [round(df[col].isnull().sum() / len(df) * 100, 2) for col in df.columns]})
display(missing_table[missing_table['missing_value_percentage'] > 0].sort_values(by=['missing_value_percentage'], ascending=False))

,columns,missing_values,missing_value_percentage
74,MiscFeature,1406,96.30
6,Alley,1369,93.77
73,Fence,1179,80.75
57,FireplaceQu,690,47.26
3,LotFrontage,259,17.74
58,GarageType,81,5.55
59,GarageYrBlt,81,5.55
60,GarageFinish,81,5.55
63,GarageQual,81,5.55
64,GarageCond,81,5.55


In [9]:
Table5= pd.crosstab(df['GarageType'].isnull(), df['MiscFeature'])
display(Table5)

MiscFeature,Gar2,Othr,Shed,TenC
GarageType,,,,
False,2,1,47,1
True,0,1,2,0


In [10]:
df.Fence = df.Fence.fillna('No Fence')

In [11]:
df.Alley = df.Alley.fillna('No Alley access')

In [12]:
table3= pd.crosstab(df['FireplaceQu'].isnull(), df['Fireplaces'])
display(table3)
print(table3)

Fireplaces,0,1,2,3
FireplaceQu,,,,
False,0,650,115,5
True,690,0,0,0


Fireplaces     0    1    2  3
FireplaceQu                  
False          0  650  115  5
True         690    0    0  0


In [13]:
df.FireplaceQu = df.FireplaceQu.fillna('No Fireplace')

In [14]:
table4= pd.crosstab(df['LotFrontage'].isnull(), df['LotConfig'])
display(table4)

LotConfig,Corner,CulDSac,FR2,FR3,Inside
LotFrontage,,,,,
False,201,45,33,4,918
True,62,49,14,0,134


In [15]:
garage_cols = ["GarageType", "GarageFinish", "GarageQual", "GarageCond"]
df[garage_cols] = df[garage_cols].fillna("No Garage")

bsmt_cols = ["BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2"]
no_bsmt = df["TotalBsmtSF"] == 0          # the 37 houses with no basement
for c in bsmt_cols:
    df.loc[no_bsmt, c] = df.loc[no_bsmt, c].fillna("No Basement")

In [16]:
df.columns

Index(['Id', 'MSSubClass', 'MSZoning', 'LotFrontage', 'LotArea', 'Street',
       'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig',
       'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType',
       'HouseStyle', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd',
       'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType',
       'MasVnrArea', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual',
       'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinSF1',
       'BsmtFinType2', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'Heating',
       'HeatingQC', 'CentralAir', 'Electrical', '1stFlrSF', '2ndFlrSF',
       'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath',
       'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'KitchenQual',
       'TotRmsAbvGrd', 'Functional', 'Fireplaces', 'FireplaceQu', 'GarageType',
       'GarageYrBlt', 'GarageFinish', 'GarageCars', 'GarageArea', 'GarageQual',
       'GarageCond', 'PavedDrive

In [17]:
y=df['SalePrice']
X=df.drop(['SalePrice', 'SaleType', 'SaleCondition', 'Id'], axis=1)
final_feature_target_table = pd.DataFrame({'Features':[X.shape[1]],'Observations':[X.shape[0]],'Target':[y.shape[0]]})
display(final_feature_target_table.style.hide(axis='index'))

Features,Observations,Target
77,1460,1460


In [18]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}, Train target shape: {y_train.shape}, Test target shape: {y_test.shape}")

Train shape: (1168, 77), Test shape: (292, 77), Train target shape: (1168,), Test target shape: (292,)


In [19]:
print(sorted(X_test.index))

[15, 23, 29, 30, 32, 43, 44, 49, 51, 56, 58, 59, 63, 65, 67, 70, 76, 78, 81, 99, 101, 107, 111, 115, 123, 128, 141, 163, 168, 175, 184, 192, 196, 198, 203, 208, 218, 220, 231, 233, 237, 239, 240, 244, 247, 259, 261, 270, 271, 274, 275, 277, 289, 297, 306, 309, 310, 316, 322, 324, 332, 339, 344, 350, 351, 353, 354, 358, 361, 363, 365, 367, 371, 374, 382, 394, 398, 411, 413, 415, 416, 422, 423, 425, 426, 427, 428, 429, 432, 433, 451, 453, 464, 478, 479, 490, 494, 522, 526, 528, 529, 532, 535, 538, 543, 554, 558, 567, 571, 575, 576, 581, 584, 585, 588, 589, 590, 591, 597, 598, 599, 602, 607, 614, 619, 620, 628, 634, 636, 643, 649, 651, 669, 670, 672, 674, 677, 679, 680, 682, 691, 693, 695, 707, 711, 718, 720, 722, 733, 736, 744, 746, 754, 755, 765, 772, 774, 777, 781, 782, 788, 796, 802, 806, 811, 812, 817, 836, 838, 843, 846, 855, 865, 874, 881, 887, 892, 894, 898, 906, 907, 910, 915, 916, 917, 922, 925, 930, 937, 939, 946, 948, 952, 958, 966, 979, 984, 989, 990, 994, 998, 1001, 1006, 10

In [22]:
X_test.index.sort_values()

Index([  15,   23,   29,   30,   32,   43,   44,   49,   51,   56,
       ...
       1410, 1414, 1421, 1427, 1428, 1430, 1432, 1447, 1449, 1450],
      dtype='int64', length=292)

## 1.1 Split features (X) from target (y)

In [27]:
summery_X = pd.concat([X_train.describe().T, X_test.describe().T], axis=1, keys=['Train', 'Test'])
Summary_y = pd.concat([y_train.describe().T, y_test.describe().T], axis=1, keys=['Train', 'Test'])
display(Summary_y.style.format("{:.2f}"))

,Train,Test
count,1168.00,292.00
mean,181441.54,178839.81
std,77263.58,87730.75
min,34900.00,35311.00
25%,130000.00,127000.00
50%,165000.00,154150.00
75%,214925.00,209175.00
max,745000.00,755000.00


## 1.1 Leak hunt: which columns would a seller not know before the sale?

### My notes

